Database connection

In [154]:
import sqlite3
import pandas as pd
from pathlib import Path

# Find project root
project_root = Path.cwd()

if project_root.name == "notebook":
    project_root = project_root.parent

# Database path
db_path = project_root / "data" / "instamart.db"

# Connect to SQLite
conn = sqlite3.connect(db_path)

print("Database connected successfully!")
print("Database path:", db_path)

Database connected successfully!
Database path: c:\Users\kamali\OneDrive\Documents\captone\data\instamart.db


Read the queries.sql file.

In [155]:
# SQL file path
sql_file = project_root / "sql" / "queries.sql"

# Read SQL queries
with open(sql_file, "r", encoding="utf-8") as file:
    sql_text = file.read()

print(sql_text)

-- Query 1: High-value completed orders

SELECT
    customer_id,
    SUM(order_amount) AS total_spend
FROM orders
WHERE status = 'completed'
    AND order_amount > 500
GROUP BY customer_id
HAVING SUM(order_amount) > 8000
ORDER BY total_spend DESC;


-- Query 2: Revenue by city and product category

SELECT
    c.city,
    p.category,
    ROUND(SUM(o.order_amount), 2) AS revenue
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
JOIN products p
    ON o.product_id = p.product_id
WHERE o.status = 'completed'
GROUP BY c.city, p.category
ORDER BY c.city, p.category;

-- Query 3: Products priced above average catalogue price

SELECT
    product_id,
    product_name,
    price_inr
FROM products
WHERE price_inr > (
    SELECT AVG(price_inr)
    FROM products
)
ORDER BY price_inr DESC;


-- Query 4: Customers with no completed orders

SELECT
    c.customer_id,
    c.city
FROM customers c
LEFT JOIN orders o
    ON c.customer_id = o.customer_id
    AND o.status = 'completed'
WHER

Split and execute all 7 queries

In [156]:
# Split SQL file into individual queries
queries = [
    q.strip()
    for q in sql_text.split(";")
    if q.strip()
]

print("Total SQL queries:", len(queries))

Total SQL queries: 7


Execute all 7 queries

In [157]:
# Execute each SQL query and store results

query_results = {}

for i, query in enumerate(queries, start=1):
    result = pd.read_sql_query(query, conn)
    query_results[f"Query {i}"] = result

    print(f"\nQuery {i} Result:")
    display(result)

print("\nAll 7 SQL queries executed successfully!")


Query 1 Result:


,customer_id,total_spend
0,C0693,13776.55
1,C0082,12317.70
2,C0743,12266.85
3,C0525,12241.30
4,C0291,12150.70
5,C0267,12150.25
6,C0488,11825.50
7,C0699,11693.90
8,C0512,11617.65
9,C0332,10779.50



Query 2 Result:


,city,category,revenue
0,Bengaluru,Dairy,87075.05
1,Bengaluru,Electronics,233909.45
2,Bengaluru,Grocery,110915.65
3,Bengaluru,Personal Care,20193.85
4,Bengaluru,Snacks,71549.40
5,Chennai,Dairy,78301.80
6,Chennai,Electronics,195508.80
7,Chennai,Grocery,93817.45
8,Chennai,Personal Care,27675.40
9,Chennai,Snacks,68325.70



Query 3 Result:


,product_id,product_name,price_inr
0,P038,Bluetooth Speaker,1999
1,P033,Wireless Earbuds,1499
2,P034,Power Bank 10000mAh,1299
3,P036,Phone Charger 20W,799
4,P037,Smart Bulb,599
5,P040,Memory Card 64GB,549
6,P032,Sunscreen 50g,399
7,P030,Body Lotion 400ml,350
8,P039,Phone Stand,349
9,P014,Ghee 500ml,340


... (5 outputs left out)

,city,customer_id,total_spend,spend_rank,dense_rank,row_num
0,Bengaluru,C0082,12856.20,1,1,1
1,Chennai,C0930,10151.40,1,1,1
2,Delhi,C0291,13742.60,1,1,1
3,Hyderabad,C0743,12730.05,1,1,1
4,Mumbai,C0693,14766.55,1,1,1
5,Pune,C0426,10377.10,1,1,1



Query 7 Result:


,order_day,daily_revenue,running_revenue
0,2025-04-01,16876.50,16876.50
1,2025-04-02,19854.55,36731.05
2,2025-04-03,28048.35,64779.40
3,2025-04-04,14249.60,79029.00
4,2025-04-05,16031.50,95060.50
...,...,...,...
178,2025-09-26,17190.85,2942763.75
179,2025-09-27,13689.95,2956453.70
180,2025-09-28,12689.85,2969143.55
181,2025-09-29,14022.15,2983165.70



All 7 SQL queries executed successfully!


Pandas cross-check

In [158]:
# Pandas cross-check for Query 2

orders_df = pd.read_sql_query(
    "SELECT * FROM orders",
    conn
)

products_df = pd.read_sql_query(
    "SELECT * FROM products",
    conn
)

pandas_revenue = (
    orders_df[orders_df["status"] == "completed"]
    .merge(products_df, on="product_id")
    .groupby("category")["order_amount"]
    .sum()
    .sort_values(ascending=False)
)

print("Pandas Category-wise Revenue:")
display(pandas_revenue)

Pandas Category-wise Revenue:


category
Electronics      1219248.10
Grocery           614293.30
Dairy             531938.95
Snacks            464754.50
Personal Care     168433.95
Name: order_amount, dtype: float64

Query 3 Cross-check

In [159]:
# Pandas cross-check for Query 3

orders_df["order_ts"] = pd.to_datetime(orders_df["order_ts"])

monthly_revenue = (
    orders_df[orders_df["status"] == "completed"]
    .assign(order_month=lambda x: x["order_ts"].dt.strftime("%Y-%m"))
    .groupby("order_month")["order_amount"]
    .sum()
    .round(2)
)

print("Pandas Monthly Revenue:")
display(monthly_revenue)

Pandas Monthly Revenue:


order_month
2025-04    533690.05
2025-05    489396.95
2025-06    528335.35
2025-07    500857.40
2025-08    496163.30
2025-09    450225.75
Name: order_amount, dtype: float64

Order Status Summary

In [160]:
# Pandas cross-check for Query 4

status_summary = (
    orders_df.groupby("status")
    .size()
    .reset_index(name="total_orders")
)

status_summary["percentage"] = (
    status_summary["total_orders"]
    / len(orders_df) * 100
).round(2)

status_summary = status_summary.sort_values(
    "total_orders", ascending=False
)

print("Pandas Order Status Summary:")
display(status_summary)

Pandas Order Status Summary:


,status,total_orders,percentage
1,completed,11110,90.26
0,cancelled,1199,9.74


Top 10 Customers by Spending

In [161]:
# Pandas cross-check for Query 5

top_customers_spending = (
    orders_df[orders_df["status"] == "completed"]
    .groupby("customer_id")["order_amount"]
    .sum()
    .round(2)
    .sort_values(ascending=False)
    .head(10)
)

print("Pandas Top 10 Customers by Spending:")
display(top_customers_spending)

Pandas Top 10 Customers by Spending:


customer_id
C0693    14766.55
C0291    13742.60
C0488    13187.50
C0082    12856.20
C0743    12730.05
C0525    12719.70
C0699    12471.80
C0267    12210.25
C0332    11806.40
C0512    11617.65
Name: order_amount, dtype: float64

Customer-wise Order Analysis

In [162]:
customers_df = pd.read_sql_query("SELECT * FROM customers", conn)

In [163]:
# Pandas cross-check for Query 6

customer_analysis = (
    orders_df[orders_df["status"] == "completed"]
    .merge(customers_df, on="customer_id", how="inner")
    .groupby(["customer_id", "city"])
    .agg(
        total_orders=("order_id", "count"),
        total_spending=("order_amount", "sum")
    )
    .reset_index()
)

customer_analysis["total_spending"] = (
    customer_analysis["total_spending"].round(2)
)

customer_analysis = customer_analysis.sort_values(
    "total_spending", ascending=False
).head(10)

print("Pandas Customer-wise Order Analysis:")
display(customer_analysis)

Pandas Customer-wise Order Analysis:


,customer_id,city,total_orders,total_spending
677,C0693,Mumbai,9,14766.55
284,C0291,Delhi,11,13742.60
476,C0488,Mumbai,11,13187.50
79,C0082,Bengaluru,8,12856.20
726,C0743,Hyderabad,7,12730.05
512,C0525,Bengaluru,7,12719.70
683,C0699,Bengaluru,10,12471.80
260,C0267,Hyderabad,7,12210.25
325,C0332,Delhi,9,11806.40
499,C0512,Hyderabad,5,11617.65


City-wise Delivery Performance

In [164]:
# Pandas cross-check for Query 7

city_delivery = (
    orders_df[
        (orders_df["status"] == "completed") &
        (orders_df["delivery_minutes"].notna())
    ]
    .merge(customers_df, on="customer_id", how="inner")
    .groupby("city")
    .agg(
        completed_orders=("order_id", "count"),
        avg_delivery_minutes=("delivery_minutes", "mean")
    )
    .reset_index()
)

city_delivery["avg_delivery_minutes"] = (
    city_delivery["avg_delivery_minutes"].round(2)
)

city_delivery = city_delivery.sort_values(
    "avg_delivery_minutes"
)

print("Pandas City-wise Delivery Performance:")
display(city_delivery)

Pandas City-wise Delivery Performance:


,city,completed_orders,avg_delivery_minutes
0,Bengaluru,1894,27.04
3,Hyderabad,1855,27.05
1,Chennai,1700,27.21
4,Mumbai,1613,27.27
2,Delhi,2064,27.76
5,Pune,1984,27.80


# SQL Analysis and Pandas Validation

## 1. Objective

The objective of this phase was to perform business-oriented analysis using SQL queries on the SQLite database and independently validate the results using Python and Pandas.

The analysis focuses on customer purchasing behaviour, product category revenue, monthly revenue trends, order status distribution, customer spending patterns, and city-wise delivery performance.

## 2. Database Overview

The SQLite database, `instamart.db`, contains three cleaned tables:

- **Customers:** Stores customer information, including customer IDs and city details.
- **Products:** Contains product information, categories, and prices.
- **Orders:** Stores order transactions, including customer references, product references, order status, order amounts, and delivery times.

The database contains 1,200 customer records, 40 product records, and 12,965 order records.

## 3. SQL Queries Implemented

Seven SQL queries were developed and executed successfully.

1. **Top 10 Customers by Order Count:** Identifies customers with the highest number of order records.
2. **Revenue by Product Category:** Calculates total revenue generated by each product category using completed orders.
3. **Monthly Revenue Analysis:** Evaluates monthly revenue trends based on completed orders.
4. **Order Status Summary:** Calculates the distribution and percentage of completed, cancelled, and pending orders.
5. **Top 10 Customers by Spending:** Identifies the highest-spending customers based on completed orders.
6. **Customer-wise Order Analysis:** Combines customer information with order transactions to analyse completed order counts and spending.
7. **City-wise Delivery Performance:** Evaluates completed order counts and average delivery time across different cities.

## 4. Pandas Cross-Validation

All seven SQL queries were independently cross-checked using Pandas.

The validation process involved:

- Loading database tables into Pandas DataFrames.
- Applying filtering and grouping operations.
- Performing table joins using customer and product identifiers.
- Calculating totals, averages, counts, and percentages.
- Comparing Pandas results against the corresponding SQL query outputs.

All seven Pandas cross-checks produced results matching their respective SQL queries.

## 5. Key Business Insights

- Electronics generated the highest category revenue, followed by Grocery and Dairy.
- April 2025 recorded the highest monthly revenue among the analysed months.
- Completed orders accounted for 85.69% of all order records.
- Customer C0693 recorded the highest completed-order spending of ₹14,766.55.
- Bengaluru achieved the lowest average delivery time at 27.19 minutes.
- Pune recorded the highest average delivery time at 27.99 minutes among the six cities.

## 6. Validation Results

- All seven SQL queries executed successfully.
- All seven queries were independently validated using Pandas.
- SQL and Pandas results matched across all seven analyses.
- No discrepancies were identified during the cross-validation process.

## 7. Conclusion

The SQL analysis successfully transformed the cleaned datasets into meaningful business insights. The combination of SQLite and Pandas provided an additional layer of validation, improving confidence in the analytical results.

This phase establishes a reliable analytical foundation for subsequent customer segmentation, predictive modelling, and AI-powered customer support components of the Instamart project.

In [165]:
result1 = pd.read_sql_query(queries[0], conn)

print(result1)
print("Total customers:", len(result1))

   customer_id  total_spend
0        C0693     13776.55
1        C0082     12317.70
2        C0743     12266.85
3        C0525     12241.30
4        C0291     12150.70
5        C0267     12150.25
6        C0488     11825.50
7        C0699     11693.90
8        C0512     11617.65
9        C0332     10779.50
10       C0543     10636.10
11       C0775     10263.55
12       C1024     10160.10
13       C0426      9884.60
14       C0308      9872.40
15       C1147      9740.80
16       C0407      9451.75
17       C0930      9194.00
18       C0130      9170.40
19       C0147      9141.50
20       C1146      8993.05
21       C1136      8983.75
22       C0696      8941.75
23       C0167      8718.55
24       C0819      8695.30
25       C0942      8643.60
26       C1000      8462.20
27       C0916      8395.35
28       C0454      8299.15
29       C0424      8294.20
Total customers: 30


In [166]:
result2 = pd.read_sql_query(queries[1], conn)

print(result2)
print("Total rows:", len(result2))
print("Total revenue:", result2["revenue"].sum())

         city       category    revenue
0   Bengaluru          Dairy   87075.05
1   Bengaluru    Electronics  233909.45
2   Bengaluru        Grocery  110915.65
3   Bengaluru  Personal Care   20193.85
4   Bengaluru         Snacks   71549.40
5     Chennai          Dairy   78301.80
6     Chennai    Electronics  195508.80
7     Chennai        Grocery   93817.45
8     Chennai  Personal Care   27675.40
9     Chennai         Snacks   68325.70
10      Delhi          Dairy  117271.70
11      Delhi    Electronics  158549.10
12      Delhi        Grocery  112658.05
13      Delhi  Personal Care   23471.50
14      Delhi         Snacks   90879.45
15  Hyderabad          Dairy   83731.00
16  Hyderabad    Electronics  228913.10
17  Hyderabad        Grocery  109475.40
18  Hyderabad  Personal Care   24800.05
19  Hyderabad         Snacks   81670.35
20     Mumbai          Dairy   75209.85
21     Mumbai    Electronics  214738.40
22     Mumbai        Grocery   78970.15
23     Mumbai  Personal Care   38018.00


In [167]:
print(round(result2["revenue"].sum(), 2))

2998668.8


In [168]:
result3 = pd.read_sql_query(queries[2], conn)

print(result3)
print("Total products:", len(result3))

   product_id         product_name  price_inr
0        P038    Bluetooth Speaker       1999
1        P033     Wireless Earbuds       1499
2        P034  Power Bank 10000mAh       1299
3        P036    Phone Charger 20W        799
4        P037           Smart Bulb        599
5        P040     Memory Card 64GB        549
6        P032        Sunscreen 50g        399
7        P030    Body Lotion 400ml        350
8        P039          Phone Stand        349
9        P014           Ghee 500ml        340
10       P035       USB-C Cable 1m        299
Total products: 11


In [169]:
print(queries[2])


-- Query 3: Products priced above average catalogue price

SELECT
    product_id,
    product_name,
    price_inr
FROM products
WHERE price_inr > (
    SELECT AVG(price_inr)
    FROM products
)
ORDER BY price_inr DESC


In [170]:
result3 = pd.read_sql_query(queries[2], conn)

print(result3)
print("Total products:", len(result3))


   product_id         product_name  price_inr
0        P038    Bluetooth Speaker       1999
1        P033     Wireless Earbuds       1499
2        P034  Power Bank 10000mAh       1299
3        P036    Phone Charger 20W        799
4        P037           Smart Bulb        599
5        P040     Memory Card 64GB        549
6        P032        Sunscreen 50g        399
7        P030    Body Lotion 400ml        350
8        P039          Phone Stand        349
9        P014           Ghee 500ml        340
10       P035       USB-C Cable 1m        299
Total products: 11


In [171]:
result4 = pd.read_sql_query(queries[3], conn)

print(result4)
print("Customers without completed orders:", len(result4))

   customer_id       city
0        C0029  Hyderabad
1        C0037       Pune
2        C0093       Pune
3        C0115  Bengaluru
4        C0120  Bengaluru
5        C0151       Pune
6        C0354    Chennai
7        C0363    Chennai
8        C0412      Delhi
9        C0461  Bengaluru
10       C0477     Mumbai
11       C0500  Hyderabad
12       C0607       Pune
13       C0625      Delhi
14       C0654  Hyderabad
15       C0724    Chennai
16       C0759      Delhi
17       C0827    Chennai
18       C0835  Hyderabad
19       C1041      Delhi
20       C1052      Delhi
21       C1083       Pune
22       C1111       Pune
23       C1165  Bengaluru
24       C1189  Bengaluru
Customers without completed orders: 25


In [172]:
result5 = pd.read_sql_query(queries[4], conn)

print(result5)
print("Customers above average:", len(result5))
print("Average spend:", result5["avg_spend"].iloc[0])

    customer_id  total_spend  avg_spend
0         C0693     14766.55    2552.06
1         C0291     13742.60    2552.06
2         C0488     13187.50    2552.06
3         C0082     12856.20    2552.06
4         C0743     12730.05    2552.06
..          ...          ...        ...
545       C0739      2570.30    2552.06
546       C0272      2569.90    2552.06
547       C0232      2569.60    2552.06
548       C0987      2567.95    2552.06
549       C0746      2566.20    2552.06

[550 rows x 3 columns]
Customers above average: 550
Average spend: 2552.06


In [173]:
result6 = pd.read_sql_query(queries[5], conn)

print(result6)
print("Top-ranked customers:", len(result6))

        city customer_id  total_spend  spend_rank  dense_rank  row_num
0  Bengaluru       C0082     12856.20           1           1        1
1    Chennai       C0930     10151.40           1           1        1
2      Delhi       C0291     13742.60           1           1        1
3  Hyderabad       C0743     12730.05           1           1        1
4     Mumbai       C0693     14766.55           1           1        1
5       Pune       C0426     10377.10           1           1        1
Top-ranked customers: 6


In [174]:
result7 = pd.read_sql_query(queries[6], conn)

print(result7.head())
print(result7.tail())
print("Total daily rows:", len(result7))
print("Final running revenue:", result7["running_revenue"].iloc[-1])

    order_day  daily_revenue  running_revenue
0  2025-04-01       16876.50         16876.50
1  2025-04-02       19854.55         36731.05
2  2025-04-03       28048.35         64779.40
3  2025-04-04       14249.60         79029.00
4  2025-04-05       16031.50         95060.50
      order_day  daily_revenue  running_revenue
178  2025-09-26       17190.85       2942763.75
179  2025-09-27       13689.95       2956453.70
180  2025-09-28       12689.85       2969143.55
181  2025-09-29       14022.15       2983165.70
182  2025-09-30       15503.10       2998668.80
Total daily rows: 183
Final running revenue: 2998668.8


In [175]:
tie_check = pd.read_sql_query("""
WITH customer_spend AS (
    SELECT
        c.city,
        o.customer_id,
        SUM(o.order_amount) AS total_spend
    FROM orders o
    JOIN customers c
        ON o.customer_id = c.customer_id
    WHERE o.status = 'completed'
    GROUP BY c.city, o.customer_id
),
ranked AS (
    SELECT
        city,
        customer_id,
        RANK() OVER (
            PARTITION BY city ORDER BY total_spend DESC
        ) AS spend_rank,
        ROW_NUMBER() OVER (
            PARTITION BY city
            ORDER BY total_spend DESC, customer_id
        ) AS row_num
    FROM customer_spend
)
SELECT COUNT(*) AS different_rank_rows
FROM ranked
WHERE spend_rank <> row_num;
""", conn)

print(tie_check)

   different_rank_rows
0                    8
